# Physics-Informed Vibration Fault Detection

**GDG KU Leuven — Team 8**

This notebook contains the cleaned public implementation of the vibration-based
structural fault-detection project developed from the Team 8 analysis.

The task is a three-class classification problem:

- `30NM` — healthy / correctly tightened fixture;
- `Loose` — loose-bolt condition;
- `Mix-45 Deg` — intermediate mixed condition.

The core methodological principle is simple:

> inspect the physics first, engineer physically meaningful spectral features,
> and evaluate models under split strategies that prevent test identity leakage.

The original experimental dataset is not redistributed in this portfolio.
Place it locally under:

```text
data/Sensor Board Update Initial Test/
```

## 1. Imports and Configuration

In [ ]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import find_peaks

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

FS = 27_000
AXES = ["X-axis", "Y-Axis", "Z-Axis"]

DATA_ROOT = Path("data") / "Sensor Board Update Initial Test"
METADATA_PATH = DATA_ROOT / "Test.csv"

if not DATA_ROOT.exists():
    print(
        "Dataset not found locally. "
        "Add the original experimental files under "
        "`data/Sensor Board Update Initial Test/` to rerun the analysis."
    )

## 2. Load Metadata and Inspect Dataset Structure

The source metadata describe each experiment, including:

- run (`A`, `B`, `C`);
- test identifier;
- bolt condition;
- excitation frequency range;
- duration and experimental setup.

Each vibration CSV contains 8,192 samples for the three accelerometer axes.

In [ ]:
if METADATA_PATH.exists():
    metadata = pd.read_csv(METADATA_PATH)

    print("Metadata shape:", metadata.shape)
    print("Columns:", list(metadata.columns))
    display(metadata.head())
else:
    metadata = None

## 3. Helper: Load One Vibration Capture

In [ ]:
def load_capture(
    run: str,
    test_id: str,
    board_id: str,
    base_path: Path = DATA_ROOT,
):
    folder = (
        base_path
        / str(run)
        / str(test_id)
        / str(board_id)
    )

    csv_files = sorted(folder.glob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(
            f"No CSV recording found in {folder}"
        )

    vibration = pd.read_csv(csv_files[0])

    if metadata is None:
        meta = None
    else:
        meta = metadata[
            (metadata["Run"] == run)
            & (
                metadata["Test"].astype(str)
                == str(test_id)
            )
        ]

    return vibration, meta

## 4. Raw Signal Inspection: Physics Before Modelling

Before feature extraction, the original analysis compares the same test and
sensor board across the three structural conditions.

The first inspection reveals an important preprocessing issue: the vertical
`Z` accelerometer contains a strong gravity-related DC offset. If it is not
removed, the FFT is dominated by the 0 Hz component and structural resonances
become difficult to compare.

The corrected spectral pipeline therefore applies:

1. mean subtraction;
2. Hann windowing;
3. FFT;
4. magnitude normalization.

In [ ]:
def compute_fft(
    signal: np.ndarray,
    fs: int = FS,
):
    x = np.asarray(signal, dtype=float)
    x = x - np.mean(x)

    window = np.hanning(len(x))

    spectrum = np.fft.rfft(
        x * window
    )

    magnitude = (
        np.abs(spectrum)
        / np.sum(window)
    )

    frequencies = np.fft.rfftfreq(
        len(x),
        d=1 / fs,
    )

    return frequencies, magnitude

In [ ]:
def find_common_capture(
    base_path: Path = DATA_ROOT,
):
    if not base_path.exists():
        return None, None

    run_a = base_path / "A"

    for test_dir in sorted(run_a.iterdir()):
        if not test_dir.is_dir():
            continue

        test_id = test_dir.name

        for board_dir in sorted(test_dir.iterdir()):
            if not board_dir.is_dir():
                continue

            board_id = board_dir.name

            exists_all = all(
                (
                    base_path
                    / run
                    / test_id
                    / board_id
                ).is_dir()
                for run in ["A", "B", "C"]
            )

            if exists_all:
                return test_id, board_id

    return None, None


COMMON_TEST_ID, COMMON_BOARD_ID = (
    find_common_capture()
)

print(
    "Example common capture:",
    COMMON_TEST_ID,
    COMMON_BOARD_ID,
)

In [ ]:
if (
    COMMON_TEST_ID is not None
    and COMMON_BOARD_ID is not None
):
    labels = {
        "A": "30NM — healthy",
        "B": "Loose",
        "C": "Mix-45°",
    }

    data = {}

    for run in ["A", "B", "C"]:
        vibration, _ = load_capture(
            run,
            COMMON_TEST_ID,
            COMMON_BOARD_ID,
        )
        data[run] = vibration

    fig, axes = plt.subplots(
        3,
        2,
        figsize=(14, 11),
    )

    for row, axis_name in enumerate(AXES):

        for run in ["A", "B", "C"]:
            x = (
                data[run][axis_name]
                .to_numpy(dtype=float)
            )

            centered = x - np.mean(x)
            t = np.arange(len(x)) / FS

            axes[row, 0].plot(
                t,
                centered,
                linewidth=0.7,
                label=labels[run],
            )

            freqs, mag = compute_fft(
                x,
                fs=FS,
            )

            axes[row, 1].plot(
                freqs,
                mag,
                linewidth=0.8,
                label=labels[run],
            )

            if run == "B":
                mask = freqs < 4000

                peaks, _ = find_peaks(
                    mag[mask],
                    height=(
                        np.max(mag[mask])
                        * 0.15
                    ),
                    distance=30,
                )

                for peak in peaks[:5]:
                    axes[row, 1].annotate(
                        f"{freqs[peak]:.0f} Hz",
                        (
                            freqs[peak],
                            mag[peak],
                        ),
                        xytext=(4, 4),
                        textcoords="offset points",
                        fontsize=7,
                    )

        axes[row, 0].set_title(
            f"{axis_name} — centered time signal"
        )
        axes[row, 0].set_xlabel("Time (s)")
        axes[row, 0].set_ylabel(
            "Centered ADC counts"
        )
        axes[row, 0].legend(fontsize=8)

        axes[row, 1].set_title(
            f"{axis_name} — normalized FFT"
        )
        axes[row, 1].set_xlim(0, 4000)
        axes[row, 1].set_xlabel(
            "Frequency (Hz)"
        )
        axes[row, 1].set_ylabel(
            "Normalized magnitude"
        )
        axes[row, 1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

### Original exploratory finding

After DC removal and normalization, the original Team 8 analysis found that
the loose condition produced sharper resonance peaks than the healthy and
mixed conditions.

Representative peaks were observed around:

- X: 168, 396, 573, 728 Hz;
- Y: 396, 570, 728 Hz;
- Z: 129, 233, 386, 570 Hz.

This directly motivated a frequency-domain feature representation rather than
a black-box classifier on raw time samples.

## 5. Physics-Informed Feature Engineering

Each recording is mapped to **36 features**:

- 10 spectral-band energies per axis;
- 1 spectral centroid per axis;
- 1 RMS feature per axis.

With three axes:

```text
3 × (10 band energies + centroid + RMS) = 36 features
```

In [ ]:
BANDS = [
    (0, 200),
    (200, 400),
    (400, 600),
    (600, 800),
    (800, 1200),
    (1200, 1600),
    (1600, 2000),
    (2000, 2500),
    (2500, 3000),
    (3000, 4000),
]


def extract_features(
    vibration: pd.DataFrame,
    fs: int = FS,
):
    features = []

    for axis in AXES:
        x = (
            vibration[axis]
            .to_numpy(dtype=float)
        )

        x = x - np.mean(x)

        window = np.hanning(len(x))

        spectrum = np.fft.rfft(
            x * window
        )

        magnitude = (
            np.abs(spectrum)
            / np.sum(window)
        )

        frequencies = np.fft.rfftfreq(
            len(x),
            d=1 / fs,
        )

        for low, high in BANDS:
            mask = (
                (frequencies >= low)
                & (frequencies < high)
            )

            band_energy = np.sum(
                magnitude[mask] ** 2
            )

            features.append(
                band_energy
            )

        centroid = (
            np.sum(
                frequencies
                * magnitude
            )
            / (
                np.sum(magnitude)
                + 1e-10
            )
        )

        rms = np.sqrt(
            np.mean(x ** 2)
        )

        features.extend(
            [centroid, rms]
        )

    return np.asarray(features)

## 6. Build the Full Feature Dataset

The original run produced **6,004 recordings**.

The labels are:

```text
30NM        -> 0
Loose       -> 1
Mix-45 Deg  -> 2
```

Each record also retains test ID, sensor board and excitation metadata so that
validation can explicitly hold out these dimensions.

In [ ]:
LABEL_MAP = {
    "30NM": 0,
    "Loose": 1,
    "Mix-45 Deg": 2,
}


def build_dataset(
    base_path: Path,
    metadata: pd.DataFrame,
):
    records = []

    for run in ["A", "B", "C"]:
        run_path = base_path / run

        if not run_path.is_dir():
            continue

        for test_path in sorted(
            run_path.iterdir()
        ):
            if not test_path.is_dir():
                continue

            test_id = test_path.name

            meta_row = metadata[
                (metadata["Run"] == run)
                & (
                    metadata["Test"].astype(str)
                    == str(test_id)
                )
            ]

            if meta_row.empty:
                continue

            bolt_status = (
                meta_row["Bolt Status"]
                .iloc[0]
            )

            if bolt_status not in LABEL_MAP:
                continue

            label = LABEL_MAP[
                bolt_status
            ]

            start_f1 = (
                meta_row["Start Freq 1"]
                .iloc[0]
            )

            end_f1 = (
                meta_row["End Freq 1"]
                .iloc[0]
            )

            start_f2 = (
                meta_row["Start Freq 2"]
                .iloc[0]
            )

            end_f2 = (
                meta_row["End Freq 2"]
                .iloc[0]
            )

            start_f1 = (
                0
                if pd.isna(start_f1)
                else start_f1
            )

            end_f1 = (
                0
                if pd.isna(end_f1)
                else end_f1
            )

            start_f2 = (
                0
                if pd.isna(start_f2)
                else start_f2
            )

            end_f2 = (
                0
                if pd.isna(end_f2)
                else end_f2
            )

            excitation = (
                f"W1:{int(start_f1)}-"
                f"{int(end_f1)}Hz"
            )

            if end_f2 > 0:
                excitation += (
                    f"_W2:{int(start_f2)}-"
                    f"{int(end_f2)}Hz"
                )

            for board_path in sorted(
                test_path.iterdir()
            ):
                if not board_path.is_dir():
                    continue

                board_id = board_path.name

                for csv_path in sorted(
                    board_path.glob("*.csv")
                ):
                    vibration = pd.read_csv(
                        csv_path
                    )

                    records.append(
                        {
                            "features": extract_features(
                                vibration
                            ),
                            "label": label,
                            "bolt_status": bolt_status,
                            "run": run,
                            "test_id": test_id,
                            "board_id": board_id,
                            "excitation": excitation,
                            "start_f1": start_f1,
                            "end_f1": end_f1,
                        }
                    )

    return records


if metadata is not None:
    records = build_dataset(
        DATA_ROOT,
        metadata,
    )

    print(
        "Feature dataset size:",
        len(records),
    )

## 7. Primary Leakage-Aware Split: By Replicate

Replicates 1–3 are used for training and replicates 4–5 for testing.

This prevents the same repeated measurement condition from appearing on both
sides of the split.

Original run:

- training samples: **3,435**;
- test samples: **2,295**.

In [ ]:
def get_replica(test_id):
    test_id = str(test_id)

    if len(test_id) == 5:
        return int(test_id[-1])

    return None


if metadata is not None:
    train_rep = [
        r
        for r in records
        if get_replica(
            r["test_id"]
        ) in [1, 2, 3]
    ]

    test_rep = [
        r
        for r in records
        if get_replica(
            r["test_id"]
        ) in [4, 5]
    ]

    print(
        "Train:",
        len(train_rep),
        "Test:",
        len(test_rep),
    )

## 8. Excitation Sensitivity

The original analysis trained an independent Logistic Regression classifier on
each excitation pattern using the replicate split.

### Reported results

| Excitation | Accuracy |
|---|---:|
| 100–1000 Hz | **1.000** |
| 50–1000 Hz | 0.988 |
| 50–200 Hz | 0.976 |
| 100–200 Hz | 0.975 |
| 50–4000 Hz | 0.945 |

The result supports the physical interpretation that the most diagnostic
structural information is concentrated in the lower-frequency resonance
region.

In [ ]:
def make_lr_classifier():
    return Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    random_state=42,
                ),
            ),
        ]
    )

In [ ]:
if metadata is not None:
    excitation_results = []

    excitations = sorted(
        {
            r["excitation"]
            for r in records
        }
    )

    for excitation in excitations:
        train_subset = [
            r
            for r in train_rep
            if (
                r["excitation"]
                == excitation
            )
        ]

        test_subset = [
            r
            for r in test_rep
            if (
                r["excitation"]
                == excitation
            )
        ]

        if (
            len(train_subset) < 10
            or len(test_subset) < 3
        ):
            continue

        y_train = np.array(
            [
                r["label"]
                for r in train_subset
            ]
        )

        if len(np.unique(y_train)) < 3:
            continue

        X_train = np.stack(
            [
                r["features"]
                for r in train_subset
            ]
        )

        X_test = np.stack(
            [
                r["features"]
                for r in test_subset
            ]
        )

        y_test = np.array(
            [
                r["label"]
                for r in test_subset
            ]
        )

        model = make_lr_classifier()
        model.fit(
            X_train,
            y_train,
        )

        accuracy = accuracy_score(
            y_test,
            model.predict(X_test),
        )

        excitation_results.append(
            {
                "excitation": excitation,
                "accuracy": accuracy,
                "n_train": len(train_subset),
                "n_test": len(test_subset),
            }
        )

    display(
        pd.DataFrame(
            excitation_results
        ).sort_values(
            "accuracy",
            ascending=False,
        )
    )

## 9. Channel Sensitivity

All combinations of the three accelerometer axes are compared using the same
replicate split.

Reported result:

| Channels | Accuracy |
|---|---:|
| X only | 0.925 |
| Y only | 0.922 |
| Z only | 0.901 |
| X + Y | 0.941 |
| X + Z | 0.951 |
| Y + Z | 0.949 |
| X + Y + Z | **0.960** |

The Z axis is weaker alone, but improves the combined model. It therefore
contains complementary rather than redundant information.

In [ ]:
AXIS_FEATURES = {
    "X only": list(range(0, 12)),
    "Y only": list(range(12, 24)),
    "Z only": list(range(24, 36)),
    "X + Y": list(range(0, 24)),
    "X + Z": (
        list(range(0, 12))
        + list(range(24, 36))
    ),
    "Y + Z": list(range(12, 36)),
    "X + Y + Z": list(range(36)),
}

## 10. Generalization Stress Test

This is the central methodological section of the project.

A naive random split can leak physical-test identity because the same test is
recorded on many boards and repeated across replicas.

The project therefore evaluates five increasingly demanding validation
strategies.

### Logistic Regression — original run

| Split | Accuracy |
|---|---:|
| Naive random | 0.966 |
| Replicate holdout | 0.960 |
| Sensor-board holdout | 0.844 |
| Excitation holdout | 0.837 |
| Combined holdout | **0.750** |

The accuracy degradation is itself an engineering result: hardware and
excitation variation are major sources of deployment uncertainty.

In [ ]:
def train_and_evaluate_lr(
    train_records,
    test_records,
):
    X_train = np.stack(
        [
            r["features"]
            for r in train_records
        ]
    )

    y_train = np.array(
        [
            r["label"]
            for r in train_records
        ]
    )

    X_test = np.stack(
        [
            r["features"]
            for r in test_records
        ]
    )

    y_test = np.array(
        [
            r["label"]
            for r in test_records
        ]
    )

    model = make_lr_classifier()

    model.fit(
        X_train,
        y_train,
    )

    predictions = model.predict(
        X_test
    )

    return {
        "model": model,
        "accuracy": accuracy_score(
            y_test,
            predictions,
        ),
        "confusion_matrix": confusion_matrix(
            y_test,
            predictions,
            labels=[0, 1, 2],
        ),
    }

In [ ]:
if metadata is not None:
    board_counts = pd.Series(
        [
            r["board_id"]
            for r in records
        ]
    ).value_counts()

    test_boards = list(
        board_counts.index[:3]
    )

    train_board = [
        r
        for r in records
        if r["board_id"]
        not in test_boards
    ]

    test_board = [
        r
        for r in records
        if r["board_id"]
        in test_boards
    ]

    excitation_counts = pd.Series(
        [
            r["excitation"]
            for r in records
        ]
    ).value_counts()

    test_excitations = list(
        excitation_counts.index[:2]
    )

    train_exc = [
        r
        for r in records
        if r["excitation"]
        not in test_excitations
    ]

    test_exc = [
        r
        for r in records
        if r["excitation"]
        in test_excitations
    ]

    train_combo = [
        r
        for r in records
        if (
            get_replica(
                r["test_id"]
            ) in [1, 2, 3]
            and r["board_id"]
            not in test_boards
            and r["excitation"]
            not in test_excitations
        )
    ]

    test_combo = [
        r
        for r in records
        if (
            get_replica(
                r["test_id"]
            ) in [4, 5]
            and r["board_id"]
            in test_boards
            and r["excitation"]
            in test_excitations
        )
    ]

## 11. Random Forest

A Random Forest is introduced to capture non-linear interactions between
frequency bands and axes.

### Original replica-split performance

```text
Accuracy = 0.9926
```

Class recall:

| Class | Recall |
|---|---:|
| 30NM | 1.00 |
| Loose | 1.00 |
| Mix-45° | 0.89 |

The intermediate class remains the hardest condition.

In [ ]:
def records_to_xy(records_subset):
    X = np.stack(
        [
            r["features"]
            for r in records_subset
        ]
    )

    y = np.array(
        [
            r["label"]
            for r in records_subset
        ]
    )

    return X, y


if metadata is not None:
    X_train_rf, y_train_rf = (
        records_to_xy(train_rep)
    )

    X_test_rf, y_test_rf = (
        records_to_xy(test_rep)
    )

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )

    rf.fit(
        X_train_rf,
        y_train_rf,
    )

    rf_predictions = rf.predict(
        X_test_rf
    )

    print(
        "Replica split accuracy:",
        accuracy_score(
            y_test_rf,
            rf_predictions,
        ),
    )

    print(
        classification_report(
            y_test_rf,
            rf_predictions,
            target_names=[
                "30NM",
                "Loose",
                "Mix-45°",
            ],
        )
    )

## 12. Logistic Regression vs Random Forest Across Splits

Reported results from the original run:

| Split | Logistic Regression | Random Forest |
|---|---:|---:|
| Naive | 0.966 | 0.995 |
| Replica | 0.960 | 0.992 |
| Sensor board | 0.844 | **0.930** |
| Excitation | 0.837 | **0.883** |
| Combined | **0.750** | 0.667 |

The non-linear model generalizes better across unseen hardware and excitation
patterns, but Logistic Regression performs better on the very small combined
holdout.

The combined test set contains only **36 observations**, so that difference
must be interpreted cautiously.

## 13. Feature Importance

The Random Forest provides a second, independent test of the physical
hypothesis.

The strongest predictors in the original run are spectral band-energy
features, especially in regions including:

- 400–600 Hz;
- 800–1200 Hz;
- 1200–1600 Hz.

This creates a useful triangulation:

```text
FFT inspection
+ excitation sensitivity
+ Random Forest feature importance
→ same physically meaningful frequency regions
```

In [ ]:
FEATURE_NAMES = []

for axis in ["X", "Y", "Z"]:
    for low, high in BANDS:
        FEATURE_NAMES.append(
            f"{axis} | {low}-{high} Hz | energy"
        )

    FEATURE_NAMES.append(
        f"{axis} | spectral centroid"
    )

    FEATURE_NAMES.append(
        f"{axis} | RMS"
    )

assert len(FEATURE_NAMES) == 36

In [ ]:
if metadata is not None:
    importances = rf.feature_importances_

    importance_df = pd.DataFrame(
        {
            "feature": FEATURE_NAMES,
            "importance": importances,
        }
    ).sort_values(
        "importance",
        ascending=False,
    )

    display(
        importance_df.head(20)
    )

    top = (
        importance_df
        .head(20)
        .sort_values("importance")
    )

    fig, ax = plt.subplots(
        figsize=(9, 7)
    )

    ax.barh(
        top["feature"],
        top["importance"],
    )

    ax.set_title(
        "Random Forest — top feature importances"
    )

    ax.set_xlabel(
        "Mean decrease in impurity"
    )

    plt.tight_layout()
    plt.show()

## 14. LightGBM and Soft-Voting Ensemble

The project also tests whether additional model complexity improves the
sensor-board holdout.

Reported results:

| Model | Board-split accuracy |
|---|---:|
| Random Forest | **0.9304** |
| LightGBM | 0.8945 |
| RF + LightGBM voting ensemble | 0.9088 |

The ensemble does not beat Random Forest, so the public project does not treat
additional complexity as automatically beneficial.

In [ ]:
try:
    import lightgbm as lgb

    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False

    print(
        "LightGBM is optional. "
        "Install `lightgbm` to rerun the final model comparison."
    )

In [ ]:
if (
    metadata is not None
    and LIGHTGBM_AVAILABLE
):
    X_train_board, y_train_board = (
        records_to_xy(train_board)
    )

    X_test_board, y_test_board = (
        records_to_xy(test_board)
    )

    rf_board = RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )

    lgbm_board = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )

    ensemble = VotingClassifier(
        estimators=[
            ("rf", rf_board),
            ("lgbm", lgbm_board),
        ],
        voting="soft",
        n_jobs=-1,
    )

    for name, model in [
        ("Random Forest", rf_board),
        ("LightGBM", lgbm_board),
        ("RF + LightGBM", ensemble),
    ]:
        model.fit(
            X_train_board,
            y_train_board,
        )

        accuracy = accuracy_score(
            y_test_board,
            model.predict(
                X_test_board
            ),
        )

        print(
            name,
            f"{accuracy:.4f}",
        )

## Key Findings

The most important conclusions from the project are methodological as much as
predictive:

- structural faults leave measurable frequency-domain signatures;
- the 100–1000 Hz excitation window is especially informative;
- all three accelerometer axes provide complementary evidence;
- naive random splitting materially overstates generalization;
- unseen sensor boards are a major source of performance degradation;
- Random Forest generalizes better than Logistic Regression on board and
  excitation holdouts;
- simpler models can still be more robust when the out-of-distribution test
  set is extremely small;
- feature importance independently confirms the physical relevance of
  specific spectral bands.

## Limitations

This is a controlled laboratory classification study.

It does not establish:

- field failure probabilities;
- remaining useful life;
- performance under arbitrary operational excitation;
- transferability to unrelated fixture geometries;
- production-level robustness to environmental noise.

The next engineering step would be broader validation across unseen hardware,
fixtures and operational conditions.